# X1 종합 예시: KAMP CNC 공정 에이전틱 봇
## S2~S4 통합 — 데이터 파이프라인 → Track A/B → Tool 함수 → ReAct 에이전트

이 노트북은 X1 과정의 핵심 예시입니다. 충청남도 A사의 CNC 공장 문제를 AI로 해결하는 전체 파이프라인을 보여줍니다.

| 섹션 | 내용 | 대응 과목 |
|------|------|-----------|
| 1 | 데이터 파이프라인 | S2 |
| 2 | Track A: 이상탐지 + RUL | Track A |
| 3 | Track B: 품질 예측 | Track B |
| 4 | CNC Tool 함수 구현 | S3 |
| 5 | ReAct 에이전트 시나리오 | **S4 핵심** |
| 6 | 의사결정 리포트 생성 | S4 |
| 7 | 내 공장 적용 가이드 | S5 연결 |

## 섹션 1: 데이터 파이프라인 (S2 내용)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import os
import warnings
warnings.filterwarnings('ignore')

import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

In [ ]:
def generate_kamp_cnc_data(machines=None, days=90, batches_per_day=20, seed=42):
    """KAMP 스타일 CNC 공정 합성 데이터 생성 (S1 노트북과 동일)"""
    if machines is None:
        machines = ['CNC-01', 'CNC-02', 'CNC-03']
    np.random.seed(seed)
    records = []
    machine_profiles = {
        'CNC-01': {'wear_rate': (0.003, 0.006), 'base_vib': 15.0, 'init_wear': 0.10},
        'CNC-02': {'wear_rate': (0.005, 0.010), 'base_vib': 20.0, 'init_wear': 0.35},
        'CNC-03': {'wear_rate': (0.003, 0.007), 'base_vib': 12.0, 'init_wear': 0.05},
    }
    maintenance_msgs = [
        '공구 인서트 교체 완료. 신품 장착.',
        '정기 PM 실시. 공구 교체 및 주축 베어링 점검.',
        '공구 파손 긴급 교체. 원인: 과부하.',
        '예방 정비 완료. 냉각수 교환 포함.',
        '공구 마모 한계 도달로 교체 조치.',
        '스핀들 진동 증가로 공구 점검 후 교체.',
    ]
    base_ts = pd.Timestamp('2024-01-01 06:00:00')

    for machine in machines:
        p = machine_profiles[machine]
        tool_wear = p['init_wear']
        base_vib = p['base_vib']
        batch_counter = 1
        for day in range(days):
            for b in range(batches_per_day):
                ts = base_ts + pd.Timedelta(days=day, minutes=b * 30)
                tool_wear = min(tool_wear + np.random.uniform(*p['wear_rate']), 1.0)
                maint = ''
                if tool_wear >= 0.85:
                    maint = np.random.choice(maintenance_msgs)
                    tool_wear = np.random.uniform(0.02, 0.08)
                elif np.random.random() < 0.02:
                    maint = '예방 정비 완료.'
                    tool_wear = np.random.uniform(0.02, 0.08)
                spindle_rpm = np.random.normal(2000, 80) * (1 - 0.05 * tool_wear)
                feed_rate = np.random.normal(150, 12)
                cutting_temp = 180 + tool_wear * 120 + np.random.normal(0, 10)
                vib_x = base_vib + tool_wear * 60 + np.random.normal(0, 3)
                vib_y = base_vib * 0.9 + tool_wear * 45 + np.random.normal(0, 2.5)
                vib_z = base_vib * 0.7 + tool_wear * 30 + np.random.normal(0, 2)
                current = 8.5 + tool_wear * 4 + np.random.normal(0, 0.5)
                if tool_wear > 0.7:
                    defect_prob = 0.25 + (tool_wear - 0.7) * 0.8
                elif tool_wear > 0.5:
                    defect_prob = 0.05 + (tool_wear - 0.5) * 0.15
                else:
                    defect_prob = 0.02
                quality_pass = 0 if np.random.random() < defect_prob else 1
                records.append({
                    'timestamp': ts, 'machine_id': machine,
                    'batch_id': f'{machine}-B{batch_counter:05d}',
                    'spindle_rpm': round(spindle_rpm, 1),
                    'feed_rate_mm_min': round(feed_rate, 1),
                    'cutting_temp_c': round(cutting_temp, 1),
                    'vibration_x_um': round(max(vib_x, 0), 2),
                    'vibration_y_um': round(max(vib_y, 0), 2),
                    'vibration_z_um': round(max(vib_z, 0), 2),
                    'current_a': round(max(current, 0), 2),
                    'tool_wear_index': round(tool_wear, 4),
                    'dimension_error_mm': round(abs(np.random.normal(0, 0.01 + tool_wear * 0.05)), 4),
                    'surface_roughness_ra': round(max(0.8 + tool_wear * 2.5 + np.random.normal(0, 0.2), 0.5), 3),
                    'quality_pass': quality_pass,
                    'maintenance_log': maint,
                })
                batch_counter += 1
    df = pd.DataFrame(records).sort_values('timestamp').reset_index(drop=True)
    return df


# 데이터 로드 (CSV 있으면 재사용, 없으면 생성)
csv_path = '../data/kamp_cnc_data.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['timestamp'])
    print(f'기존 데이터 로드: {len(df):,}행')
else:
    os.makedirs('../data', exist_ok=True)
    df = generate_kamp_cnc_data()
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'데이터 생성 및 저장: {len(df):,}행')

print(f'기간: {df.timestamp.min().date()} ~ {df.timestamp.max().date()}')
print(f'설비: {df.machine_id.unique().tolist()}')

In [ ]:
# 전처리: 결측값 확인 + 시계열 분할
print('=== 결측값 현황 ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '결측값 없음 ✓')

# 훈련/검증 분할 (마지막 14일을 검증 세트로)
split_date = df.timestamp.max() - pd.Timedelta(days=14)
df_train = df[df.timestamp <= split_date].copy()
df_test  = df[df.timestamp > split_date].copy()

print(f'\n훈련 세트: {len(df_train):,}행 ({df_train.timestamp.min().date()} ~ {df_train.timestamp.max().date()})')
print(f'검증 세트: {len(df_test):,}행  ({df_test.timestamp.min().date()} ~ {df_test.timestamp.max().date()})')

# 특성 스케일링용 통계 (훈련 세트 기준)
FEATURE_COLS = ['spindle_rpm', 'feed_rate_mm_min', 'cutting_temp_c',
                'vibration_x_um', 'vibration_y_um', 'vibration_z_um', 'current_a']
feat_mean = df_train[FEATURE_COLS].mean()
feat_std  = df_train[FEATURE_COLS].std()
print('\n특성 스케일링 통계 저장 완료')

In [ ]:
# EDA: 공구 마모 진행에 따른 센서 변화 시각화
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

colors = {'CNC-01': '#2196F3', 'CNC-02': '#F44336', 'CNC-03': '#4CAF50'}

sensor_cols = [
    ('vibration_x_um', '진동 X (µm)'),
    ('cutting_temp_c', '절삭 온도 (°C)'),
    ('current_a', '전류 (A)'),
    ('surface_roughness_ra', '표면 거칠기 Ra'),
    ('dimension_error_mm', '치수 오차 (mm)'),
]

sample = df.sample(min(2000, len(df)), random_state=42)

for ax, (col, label) in zip(axes[:5], sensor_cols):
    for machine, color in colors.items():
        mdata = sample[sample.machine_id == machine]
        ax.scatter(mdata.tool_wear_index, mdata[col],
                   alpha=0.3, s=10, color=color, label=machine)
    ax.set_xlabel('공구 마모 지수')
    ax.set_ylabel(label)
    ax.set_title(f'마모 vs {label}')
    ax.axvline(x=0.7, color='orange', linestyle='--', alpha=0.7, label='경고(0.7)')
    ax.legend(fontsize=7, markerscale=3)

# 6번째: 설비별 일별 불량률 추이
df['date'] = df.timestamp.dt.date
daily_defect = df.groupby(['date', 'machine_id'])['quality_pass'].apply(
    lambda x: (1 - x.mean()) * 100).reset_index()
daily_defect.columns = ['date', 'machine_id', 'defect_rate']

for machine, color in colors.items():
    mdata = daily_defect[daily_defect.machine_id == machine]
    axes[5].plot(range(len(mdata)), mdata.defect_rate.values,
                 color=color, label=machine, alpha=0.8, linewidth=1.5)
axes[5].axhline(y=8, color='red', linestyle='--', label='현재 평균 8%')
axes[5].axhline(y=3, color='green', linestyle='--', label='목표 3%')
axes[5].set_title('설비별 일별 불량률 추이 (90일)')
axes[5].set_xlabel('경과 일수')
axes[5].set_ylabel('불량률 (%)')
axes[5].legend(fontsize=8)

plt.suptitle('KAMP CNC 공정 EDA — 공구 마모와 품질/센서 관계', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kamp_cnc_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ 공구 마모가 증가할수록 진동, 온도, 불량률이 모두 증가하는 명확한 패턴 확인')

## 섹션 2: Track A 방식 — 이상탐지 + RUL 예측

In [ ]:
# Track A: 임계값 기반 이상탐지
# 실제 Track A에서는 FFT + Autoencoder를 사용하지만,
# 여기서는 원리를 보여주기 위해 통계적 임계값을 사용합니다.

def detect_anomaly(row, thresholds):
    """임계값 기반 이상탐지"""
    anomalies = []
    if row.vibration_x_um > thresholds['vib_x']:
        anomalies.append(f'진동X 초과({row.vibration_x_um:.1f} > {thresholds["vib_x"]})')
    if row.cutting_temp_c > thresholds['temp']:
        anomalies.append(f'온도 초과({row.cutting_temp_c:.1f} > {thresholds["temp"]})')
    if row.current_a > thresholds['current']:
        anomalies.append(f'전류 초과({row.current_a:.1f} > {thresholds["current"]})')
    if row.tool_wear_index > thresholds['wear_warn']:
        anomalies.append(f'마모 경고({row.tool_wear_index:.2f} > {thresholds["wear_warn"]})')
    return 'ANOMALY' if anomalies else 'NORMAL', anomalies

# 정상 구간(마모 < 0.4) 기준으로 임계값 설정 (훈련 세트)
normal_df = df_train[df_train.tool_wear_index < 0.4]
THRESHOLDS = {
    'vib_x':    normal_df.vibration_x_um.mean() + 3 * normal_df.vibration_x_um.std(),
    'temp':     normal_df.cutting_temp_c.mean() + 3 * normal_df.cutting_temp_c.std(),
    'current':  normal_df.current_a.mean() + 3 * normal_df.current_a.std(),
    'wear_warn': 0.70,
    'wear_crit': 0.85,
}

print('설정된 이상탐지 임계값:')
for k, v in THRESHOLDS.items():
    print(f'  {k}: {v:.2f}')

In [ ]:
# Track A: RUL (잔여 유용 수명) 예측
# 현재 마모 증가율을 기반으로 공구 교체까지 남은 배치 수 추정

def estimate_rul(machine_id, df, threshold_wear=0.85, recent_n=20):
    """
    최근 N개 배치의 마모 증가율로 RUL 추정
    Returns: {'current_wear': float, 'wear_rate_per_batch': float,
              'remaining_batches': int, 'remaining_hours': float, 'urgency': str}
    """
    mdf = df[df.machine_id == machine_id].sort_values('timestamp').tail(recent_n)
    if len(mdf) < 2:
        return None
    
    current_wear = mdf.tool_wear_index.iloc[-1]
    # 마모 증가율 계산 (선형 회귀 기울기)
    x = np.arange(len(mdf))
    y = mdf.tool_wear_index.values
    coeffs = np.polyfit(x, y, 1)
    wear_rate = max(coeffs[0], 0.001)  # 음수 방지
    
    remaining_batches = max(0, int((threshold_wear - current_wear) / wear_rate))
    remaining_hours = remaining_batches * 0.5  # 1배치 = 30분
    
    if remaining_hours < 4:
        urgency = 'CRITICAL'
    elif remaining_hours < 12:
        urgency = 'HIGH'
    elif remaining_hours < 24:
        urgency = 'MEDIUM'
    else:
        urgency = 'LOW'
    
    return {
        'machine_id': machine_id,
        'current_wear': round(current_wear, 3),
        'wear_rate_per_batch': round(wear_rate, 5),
        'remaining_batches': remaining_batches,
        'remaining_hours': round(remaining_hours, 1),
        'urgency': urgency,
    }

# 각 설비의 RUL 계산
print('=== 현재 시점 RUL 예측 결과 ===')
rul_results = {}
for machine in ['CNC-01', 'CNC-02', 'CNC-03']:
    result = estimate_rul(machine, df)
    rul_results[machine] = result
    print(f"\n{machine}:")
    print(f"  현재 마모: {result['current_wear']} ({result['urgency']})")
    print(f"  잔여 배치: {result['remaining_batches']}개")
    print(f"  잔여 시간: {result['remaining_hours']}시간")

# 이상 신호 저장
os.makedirs('../outputs', exist_ok=True)
with open('../outputs/rul_signal.json', 'w', encoding='utf-8') as f:
    json.dump(rul_results, f, ensure_ascii=False, indent=2)
print('\n→ rul_signal.json 저장 완료')

In [ ]:
# 이상탐지 결과 생성 및 저장
recent_data = df[df.timestamp >= df.timestamp.max() - pd.Timedelta(hours=24)]
anomaly_summary = {}
for machine in ['CNC-01', 'CNC-02', 'CNC-03']:
    mdf = recent_data[recent_data.machine_id == machine]
    anomaly_count = 0
    for _, row in mdf.iterrows():
        status, _ = detect_anomaly(row, THRESHOLDS)
        if status == 'ANOMALY':
            anomaly_count += 1
    total = len(mdf)
    anomaly_summary[machine] = {
        'total_batches': total,
        'anomaly_batches': anomaly_count,
        'anomaly_rate': round(anomaly_count / total, 3) if total > 0 else 0,
        'status': 'ANOMALY' if anomaly_count / total > 0.3 else 'NORMAL'
    }

with open('../outputs/anomaly_signal.json', 'w', encoding='utf-8') as f:
    json.dump(anomaly_summary, f, ensure_ascii=False, indent=2)

print('=== 최근 24시간 이상탐지 결과 ===')
for m, s in anomaly_summary.items():
    print(f"{m}: 이상 {s['anomaly_batches']}/{s['total_batches']}배치 ({s['anomaly_rate']*100:.0f}%) — {s['status']}")

## 섹션 3: Track B 방식 — 품질 예측

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import classification_report, roc_auc_score
    from sklearn.preprocessing import StandardScaler
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print('scikit-learn 미설치 — 규칙 기반 대체 모델 사용')

QUALITY_FEATURES = [
    'spindle_rpm', 'feed_rate_mm_min', 'cutting_temp_c',
    'vibration_x_um', 'vibration_y_um', 'vibration_z_um',
    'current_a', 'tool_wear_index'
]

if SKLEARN_AVAILABLE:
    X_train = df_train[QUALITY_FEATURES].values
    y_train = df_train['quality_pass'].values
    X_test  = df_test[QUALITY_FEATURES].values
    y_test  = df_test['quality_pass'].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf_model.fit(X_train_s, y_train)

    y_pred = rf_model.predict(X_test_s)
    y_prob = rf_model.predict_proba(X_test_s)[:, 0]  # 불량 확률

    print('=== Track B: Random Forest 품질 예측 모델 ===')
    print(classification_report(y_test, y_pred, target_names=['불합격', '합격']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}')

    # Feature Importance
    importances = pd.Series(rf_model.feature_importances_, index=QUALITY_FEATURES)
    importances = importances.sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 4))
    importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Track B: 품질 예측 모델 — Feature Importance')
    ax.set_ylabel('중요도')
    ax.set_xlabel('공정 파라미터')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'\n가장 중요한 파라미터: {importances.index[0]} ({importances.iloc[0]:.3f})')
    print('→ 공구 마모 지수가 불량 예측에서 가장 중요한 변수 (Track A와 일치)')

else:
    # scikit-learn 없을 때 규칙 기반 대체
    def predict_quality_rule(row):
        defect_prob = 0.02
        if row.tool_wear_index > 0.7:
            defect_prob += 0.25 + (row.tool_wear_index - 0.7) * 0.8
        if row.cutting_temp_c > 260:
            defect_prob += 0.1
        return min(defect_prob, 0.95)
    print('규칙 기반 모델 준비 완료')
    scaler = None
    rf_model = None

In [ ]:
# 불량 신호 저장
defect_summary = {}
for machine in ['CNC-01', 'CNC-02', 'CNC-03']:
    mdf = recent_data[recent_data.machine_id == machine]
    defect_rate = 1 - mdf.quality_pass.mean()
    defect_summary[machine] = {
        'defect_rate_24h': round(defect_rate, 3),
        'defect_count_24h': int((mdf.quality_pass == 0).sum()),
        'total_batches_24h': len(mdf),
        'status': 'HIGH_RISK' if defect_rate > 0.10 else ('WARNING' if defect_rate > 0.05 else 'NORMAL')
    }

with open('../outputs/defect_signal.json', 'w', encoding='utf-8') as f:
    json.dump(defect_summary, f, ensure_ascii=False, indent=2)

print('=== 최근 24시간 불량 현황 ===')
for m, s in defect_summary.items():
    print(f"{m}: 불량률 {s['defect_rate_24h']*100:.1f}% ({s['defect_count_24h']}/{s['total_batches_24h']}배치) — {s['status']}")

## 섹션 4: CNC 특화 Tool 함수 구현 (S3 내용)

에이전트가 호출할 수 있는 Tool 함수들을 구현합니다.  
실제 환경에서는 이 함수들이 실시간 DB를 조회하게 됩니다.

In [ ]:
# =============================================================================
# CNC Tool 함수 모음
# =============================================================================

def get_cnc_status(machine_id='CNC-01'):
    """
    현재 공구 마모 상태 + 이상 여부 조회
    
    Args:
        machine_id: 설비 ID ('CNC-01', 'CNC-02', 'CNC-03')
    Returns:
        dict: 현재 상태 정보
    """
    mdf = df[df.machine_id == machine_id].sort_values('timestamp')
    if len(mdf) == 0:
        return {'error': f'설비 {machine_id} 데이터 없음'}
    
    latest = mdf.iloc[-1]
    status, anomaly_reasons = detect_anomaly(latest, THRESHOLDS)
    
    wear = latest.tool_wear_index
    if wear >= THRESHOLDS['wear_crit']:
        wear_level = 'CRITICAL'
    elif wear >= THRESHOLDS['wear_warn']:
        wear_level = 'WARNING'
    elif wear >= 0.5:
        wear_level = 'CAUTION'
    else:
        wear_level = 'NORMAL'
    
    return {
        'machine_id': machine_id,
        'timestamp': str(latest.timestamp),
        'tool_wear_index': round(wear, 3),
        'wear_level': wear_level,
        'anomaly_status': status,
        'anomaly_reasons': anomaly_reasons,
        'current_spindle_rpm': round(latest.spindle_rpm, 0),
        'current_temp_c': round(latest.cutting_temp_c, 1),
        'current_vibration_x': round(latest.vibration_x_um, 1),
        'last_maintenance': mdf[mdf.maintenance_log != ''].iloc[-1].maintenance_log if len(mdf[mdf.maintenance_log != '']) > 0 else '기록 없음',
    }


def get_cnc_rul(machine_id='CNC-01'):
    """
    공구 교체까지 잔여 배치 수 + 일수 예측
    
    Args:
        machine_id: 설비 ID
    Returns:
        dict: RUL 예측 결과
    """
    result = estimate_rul(machine_id, df)
    if result is None:
        return {'error': '데이터 부족'}
    return result


def get_quality_prediction(machine_id, spindle_rpm=None, feed_rate=None, cutting_temp=None):
    """
    입력 파라미터로 다음 배치 불량 예측
    
    Args:
        machine_id: 설비 ID
        spindle_rpm: 스핀들 속도 (미입력 시 현재값 사용)
        feed_rate: 이송속도
        cutting_temp: 절삭 온도
    Returns:
        dict: 불량 예측 결과
    """
    mdf = df[df.machine_id == machine_id].sort_values('timestamp')
    latest = mdf.iloc[-1]
    
    # 미입력 파라미터는 현재값 사용
    if spindle_rpm is None:
        spindle_rpm = latest.spindle_rpm
    if feed_rate is None:
        feed_rate = latest.feed_rate_mm_min
    if cutting_temp is None:
        cutting_temp = latest.cutting_temp_c
    
    tool_wear = latest.tool_wear_index
    
    if SKLEARN_AVAILABLE and rf_model is not None:
        features = np.array([[
            spindle_rpm, feed_rate, cutting_temp,
            latest.vibration_x_um, latest.vibration_y_um, latest.vibration_z_um,
            latest.current_a, tool_wear
        ]])
        features_s = scaler.transform(features)
        defect_prob = rf_model.predict_proba(features_s)[0][0]  # 불량 확률
        method = 'Random Forest'
    else:
        # 규칙 기반
        defect_prob = 0.02
        if tool_wear > 0.7:
            defect_prob += 0.25 + (tool_wear - 0.7) * 0.8
        if cutting_temp > 260:
            defect_prob += 0.1
        defect_prob = min(defect_prob, 0.95)
        method = 'Rule-based'
    
    risk = 'HIGH' if defect_prob > 0.15 else ('MEDIUM' if defect_prob > 0.07 else 'LOW')
    
    return {
        'machine_id': machine_id,
        'defect_probability': round(defect_prob, 3),
        'defect_probability_pct': f'{defect_prob*100:.1f}%',
        'risk_level': risk,
        'prediction_method': method,
        'input_params': {
            'spindle_rpm': round(spindle_rpm, 0),
            'feed_rate_mm_min': round(feed_rate, 1),
            'cutting_temp_c': round(cutting_temp, 1),
            'tool_wear_index': round(tool_wear, 3),
        }
    }


def get_cnc_statistics(machine_id, period_hours=168):
    """
    지정 기간 통계: 이상 발생률, 평균 공구 마모, 불량률
    
    Args:
        machine_id: 설비 ID
        period_hours: 조회 기간 (기본 168시간 = 7일)
    Returns:
        dict: 통계 요약
    """
    cutoff = df.timestamp.max() - pd.Timedelta(hours=period_hours)
    mdf = df[(df.machine_id == machine_id) & (df.timestamp >= cutoff)]
    
    if len(mdf) == 0:
        return {'error': '데이터 없음'}
    
    anomaly_count = sum(
        1 for _, row in mdf.iterrows()
        if detect_anomaly(row, THRESHOLDS)[0] == 'ANOMALY'
    )
    
    wear_trend = 'degrading' if mdf.tool_wear_index.iloc[-1] > mdf.tool_wear_index.iloc[0] else 'stable'
    
    return {
        'machine_id': machine_id,
        'period_hours': period_hours,
        'total_batches': len(mdf),
        'anomaly_batches': anomaly_count,
        'anomaly_rate': round(anomaly_count / len(mdf), 3),
        'avg_tool_wear': round(mdf.tool_wear_index.mean(), 3),
        'wear_start': round(mdf.tool_wear_index.iloc[0], 3),
        'wear_end': round(mdf.tool_wear_index.iloc[-1], 3),
        'trend': wear_trend,
        'avg_defect_rate': round(1 - mdf.quality_pass.mean(), 3),
        'total_defects': int((mdf.quality_pass == 0).sum()),
        'maintenance_events': int((mdf.maintenance_log != '').sum()),
    }


def get_optimal_parameters(machine_id):
    """
    ML이 찾은 최적 가공 파라미터 제안
    
    Args:
        machine_id: 설비 ID
    Returns:
        dict: 최적 파라미터 및 예상 효과
    """
    # 불량 없는 구간에서 파라미터 통계
    good_batches = df[(df.machine_id == machine_id) & (df.quality_pass == 1) & (df.tool_wear_index < 0.5)]
    mdf = df[df.machine_id == machine_id].sort_values('timestamp')
    current_wear = mdf.tool_wear_index.iloc[-1]
    
    if len(good_batches) == 0:
        return {'error': '최적화 데이터 부족'}
    
    # 마모가 높을 때 스핀들 속도를 낮추면 수명 연장
    wear_factor = current_wear / 0.85
    spindle_reduction = min(int(wear_factor * 15), 20)  # 최대 20% 감속
    
    return {
        'machine_id': machine_id,
        'current_wear': round(current_wear, 3),
        'recommended_spindle_rpm': round(good_batches.spindle_rpm.mean() * (1 - spindle_reduction/100), 0),
        'recommended_feed_rate': round(good_batches.feed_rate_mm_min.mean() * 0.95, 1),
        'recommended_temp_target': round(good_batches.cutting_temp_c.mean(), 1),
        'reduce_spindle_by': f'{spindle_reduction}%',
        'expected_extension_hours': round(spindle_reduction * 0.2, 1),
        'expected_defect_reduction': f'{min(spindle_reduction * 2, 30)}%',
        'rationale': f'마모도 {current_wear:.0%} → 스핀들 {spindle_reduction}% 감속으로 절삭력 감소, 수명 연장'
    }


# Tool 함수 테스트
print('=== Tool 함수 테스트 ===')
print('\n[get_cnc_status CNC-01]')
import json
print(json.dumps(get_cnc_status('CNC-01'), ensure_ascii=False, indent=2))

In [ ]:
print('[get_cnc_rul CNC-02]')
print(json.dumps(get_cnc_rul('CNC-02'), ensure_ascii=False, indent=2))

In [ ]:
print('[get_quality_prediction CNC-01]')
print(json.dumps(get_quality_prediction('CNC-01'), ensure_ascii=False, indent=2))
print()
print('[get_cnc_statistics CNC-02, 24시간]')
print(json.dumps(get_cnc_statistics('CNC-02', 24), ensure_ascii=False, indent=2))

In [ ]:
print('[get_optimal_parameters CNC-01]')
print(json.dumps(get_optimal_parameters('CNC-01'), ensure_ascii=False, indent=2))

## 섹션 5: ReAct 에이전트 시나리오

### ReAct 패턴이란?

**Re**asoning + **Act**ing — LLM이 추론과 행동을 번갈아 수행하며 문제를 해결합니다.

```
질문 → Thought(추론) → Action(Tool 실행) → Observation(결과 확인) → ... → Finish(최종 답변)
```

실제 구현에서는 LLM API를 사용하지만, 여기서는 에이전트 추론 과정을 `print()`로 시뮬레이션합니다.

In [ ]:
def simulate_react_agent(user_question, verbose=True):
    """
    ReAct 에이전트 시뮬레이션
    실제 환경에서는 LLM이 Thought/Action을 생성하지만,
    여기서는 CNC-01 야간교대 시나리오를 시뮬레이션합니다.
    """
    # 질문에서 설비 ID 추출 (간단 파싱)
    machine_id = 'CNC-01'
    for m in ['CNC-01', 'CNC-02', 'CNC-03']:
        if m in user_question:
            machine_id = m
            break
    
    print('=' * 65)
    print(f'[사용자] {user_question}')
    print('=' * 65)
    
    steps = []
    
    # Step 1
    thought1 = f"{machine_id} 현재 상태와 지난 통계를 먼저 확인해야겠다."
    action1 = f"get_cnc_status('{machine_id}')"
    obs1 = get_cnc_status(machine_id)
    steps.append((thought1, action1, obs1))
    
    if verbose:
        print(f"\nThought: {thought1}")
        print(f"Action:  {action1}")
        print(f"Observation: 공구마모={obs1['tool_wear_index']} ({obs1['wear_level']}), "
              f"이상={obs1['anomaly_status']}, "
              f"온도={obs1['current_temp_c']}°C")
    
    # Step 2
    thought2 = (
        f"공구 마모가 {obs1['tool_wear_index']} ({obs1['wear_level']}). "
        f"잔여 배치 수 확인이 필요하다."
    )
    action2 = f"get_cnc_rul('{machine_id}')"
    obs2 = get_cnc_rul(machine_id)
    steps.append((thought2, action2, obs2))
    
    if verbose:
        print(f"\nThought: {thought2}")
        print(f"Action:  {action2}")
        print(f"Observation: 잔여배치={obs2['remaining_batches']}개, "
              f"잔여시간={obs2['remaining_hours']}시간, "
              f"긴급도={obs2['urgency']}")
    
    # Step 3
    thought3 = "불량률 추세와 이상 발생률 통계도 확인하자."
    action3 = f"get_cnc_statistics('{machine_id}', 24)"
    obs3 = get_cnc_statistics(machine_id, 24)
    steps.append((thought3, action3, obs3))
    
    if verbose:
        print(f"\nThought: {thought3}")
        print(f"Action:  {action3}")
        print(f"Observation: 이상률={obs3['anomaly_rate']*100:.0f}%, "
              f"불량률={obs3['avg_defect_rate']*100:.1f}%, "
              f"추세={obs3['trend']}")
    
    # Step 4
    thought4 = "최적 파라미터로 속도 감속 시 야간 교대까지 버틸 수 있는지 확인하자."
    action4 = f"get_optimal_parameters('{machine_id}')"
    obs4 = get_optimal_parameters(machine_id)
    steps.append((thought4, action4, obs4))
    
    if verbose:
        print(f"\nThought: {thought4}")
        print(f"Action:  {action4}")
        print(f"Observation: 스핀들 {obs4['reduce_spindle_by']} 감속 → "
              f"{obs4['expected_extension_hours']}시간 연장 가능")
    
    # 최종 답변 생성
    wear = obs1['tool_wear_index']
    hours_left = obs2['remaining_hours']
    urgency = obs2['urgency']
    defect_rate = obs3['avg_defect_rate']
    reduction = obs4['reduce_spindle_by']
    extension = obs4['expected_extension_hours']
    
    final_answer = f"""
Finish:

[{machine_id} 상태 요약]
  공구 마모: {wear} ({obs1['wear_level']}) {'⚠️ 임계값 초과!' if wear > 0.7 else '✓'}
  잔여 시간: 약 {hours_left}시간 ({urgency})
  24시간 불량률: {defect_rate*100:.1f}%

[권고 조치]
  ① 가능하면 교대 전 공구 교체 권장 (약 15분 소요)
  ② 즉시 교체 불가 시: 스핀들 속도 {reduction} 감속 → {extension}시간 연장 가능
  ③ 파라미터 최적화 병행하면 불량률 {obs4['expected_defect_reduction']} 추가 개선 기대

[우선순위]
  {"🔴 즉시 조치 필요" if urgency in ['CRITICAL','HIGH'] else "🟡 금일 내 조치 권장"}
"""
    print(final_answer)
    
    return {
        'machine_id': machine_id,
        'steps': len(steps),
        'tools_called': [a for _, a, _ in steps],
        'final_wear': wear,
        'urgency': urgency,
        'hours_left': hours_left,
    }


# 시나리오 실행
result = simulate_react_agent(
    "CNC-01 지금 상태 어때요? 오늘 야간 교대 전에 확인해야 할 게 있나요?"
)

In [ ]:
# 시나리오 2: 설비 비교 질문
def simulate_multi_machine_query(verbose=True):
    """3대 설비 종합 상태 조회 시나리오"""
    print('=' * 65)
    print('[사용자] 오늘 오전 라인 점검 결과 알려주세요. 3대 모두 확인해주세요.')
    print('=' * 65)
    
    print('\nThought: 3대 설비 순서대로 상태와 RUL을 확인해야겠다.')
    
    results = []
    for machine in ['CNC-01', 'CNC-02', 'CNC-03']:
        status = get_cnc_status(machine)
        rul = get_cnc_rul(machine)
        stats = get_cnc_statistics(machine, 24)
        results.append((machine, status, rul, stats))
        print(f"Action:  get_cnc_status('{machine}'), get_cnc_rul('{machine}')")
        print(f"  → 마모={status['tool_wear_index']}, 잔여={rul['remaining_hours']}h, "
              f"불량률={stats['avg_defect_rate']*100:.1f}%, 긴급도={rul['urgency']}")
    
    # 위험도 순 정렬
    urgency_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    results.sort(key=lambda x: urgency_order.get(x[2]['urgency'], 4))
    
    print('\nFinish:')
    print('\n[오늘 오전 라인 점검 결과] (위험도 순)')
    print('-' * 55)
    for machine, status, rul, stats in results:
        icon = '🔴' if rul['urgency'] in ['CRITICAL','HIGH'] else ('🟡' if rul['urgency'] == 'MEDIUM' else '🟢')
        print(f"{icon} {machine}: 마모 {status['tool_wear_index']:.0%}, "
              f"잔여 {rul['remaining_hours']}h, "
              f"불량률 {stats['avg_defect_rate']*100:.1f}%, "
              f"긴급도 {rul['urgency']}")
    
    most_urgent = results[0][0]
    print(f'\n→ 최우선 관리 설비: {most_urgent} (즉시 점검 권고)')

simulate_multi_machine_query()

## 섹션 6: 의사결정 리포트 자동 생성

In [ ]:
def generate_equipment_health_report(machine_id, output_format='markdown'):
    """
    설비 건강 리포트 자동 생성
    
    Args:
        machine_id: 설비 ID
        output_format: 'markdown' 또는 'text'
    Returns:
        str: 리포트 텍스트
    """
    status = get_cnc_status(machine_id)
    rul = get_cnc_rul(machine_id)
    stats_7d = get_cnc_statistics(machine_id, 168)
    stats_24h = get_cnc_statistics(machine_id, 24)
    opt = get_optimal_parameters(machine_id)
    
    from datetime import datetime
    now = datetime.now().strftime('%Y-%m-%d %H:%M')
    
    report = f"""# {machine_id} 설비 건강 리포트
생성 시각: {now} | 기준 데이터: 최근 7일 + 현재 상태

---

## 1. 현재 상태 요약

| 항목 | 값 | 상태 |
|------|----|------|
| 공구 마모 지수 | {status['tool_wear_index']} | {status['wear_level']} |
| 이상 감지 | {status['anomaly_status']} | {'주의 필요' if status['anomaly_status'] == 'ANOMALY' else '정상 범위'} |
| 스핀들 속도 | {status['current_spindle_rpm']} RPM | - |
| 절삭 온도 | {status['current_temp_c']} °C | - |
| 진동 (X축) | {status['current_vibration_x']} µm | - |
| 마지막 정비 | {status['last_maintenance'][:30] if len(status['last_maintenance']) > 30 else status['last_maintenance']} | - |

## 2. 공구 수명 예측 (RUL)

- **잔여 배치**: {rul['remaining_batches']}개
- **잔여 시간**: 약 {rul['remaining_hours']}시간
- **긴급도**: **{rul['urgency']}**
- **마모 증가율**: 배치당 {rul['wear_rate_per_batch']:.5f}

## 3. 최근 7일 성능 통계

| 항목 | 7일 평균 | 최근 24시간 |
|------|---------|------------|
| 불량률 | {stats_7d['avg_defect_rate']*100:.1f}% | {stats_24h['avg_defect_rate']*100:.1f}% |
| 이상 발생률 | {stats_7d['anomaly_rate']*100:.0f}% | {stats_24h['anomaly_rate']*100:.0f}% |
| 불량 건수 | {stats_7d['total_defects']}건 | {stats_24h['total_defects']}건 |
| 정비 이벤트 | {stats_7d['maintenance_events']}회 | {stats_24h['maintenance_events']}회 |

## 4. 최적화 권고

- 스핀들 속도: {opt['recommended_spindle_rpm']} RPM (현재 대비 {opt['reduce_spindle_by']} 감속)
- 이송 속도: {opt['recommended_feed_rate']} mm/min
- **예상 효과**: 수명 {opt['expected_extension_hours']}시간 연장, 불량률 {opt['expected_defect_reduction']} 감소
- **근거**: {opt['rationale']}

## 5. 권고 조치

"""
    if rul['urgency'] == 'CRITICAL':
        report += """> **🔴 즉시 조치 필요**
> 1. 현재 작업 완료 후 즉시 공구 교체 (약 15분)
> 2. 교체 전 스핀들 속도 20% 감속 운영
> 3. 교체 후 첫 5배치 품질 집중 모니터링
"""
    elif rul['urgency'] == 'HIGH':
        report += """> **🟠 금일 내 조치 권고**
> 1. 이번 교대 내 공구 교체 계획 수립
> 2. 스핀들 속도 15% 감속으로 수명 연장
> 3. 불량률 모니터링 주기 단축 (매 10배치)
"""
    else:
        report += """> **🟢 정상 운영**
> 1. 정기 모니터링 유지
> 2. 다음 예방 정비 일정 확인
"""
    report += f"\n---\n*생성: CNC 에이전틱 봇 v1.0 | X1 KAMP CNC 예시*"
    return report


# CNC-02 (가장 위험한 설비) 리포트 생성
report = generate_equipment_health_report('CNC-02')
print(report)

In [ ]:
# 시각화: 3대 설비 종합 대시보드
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'CNC-01': '#2196F3', 'CNC-02': '#F44336', 'CNC-03': '#4CAF50'}

# 1. 공구 마모 추이 (30일)
ax = axes[0, 0]
recent_30 = df[df.timestamp >= df.timestamp.max() - pd.Timedelta(days=30)]
for machine, color in colors.items():
    mdf = recent_30[recent_30.machine_id == machine]
    daily = mdf.groupby(mdf.timestamp.dt.date)['tool_wear_index'].mean()
    ax.plot(range(len(daily)), daily.values, color=color, label=machine, linewidth=2)
ax.axhline(y=0.70, color='orange', linestyle='--', alpha=0.8, label='경고(0.70)')
ax.axhline(y=0.85, color='red', linestyle='--', alpha=0.8, label='교체(0.85)')
ax.set_title('공구 마모 추이 (최근 30일)', fontweight='bold')
ax.set_xlabel('경과 일수')
ax.set_ylabel('공구 마모 지수')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# 2. 설비별 현재 공구 마모 막대
ax = axes[0, 1]
machines = ['CNC-01', 'CNC-02', 'CNC-03']
wear_values = [get_cnc_status(m)['tool_wear_index'] for m in machines]
bar_colors = ['red' if w > 0.7 else ('orange' if w > 0.5 else 'green') for w in wear_values]
bars = ax.bar(machines, wear_values, color=bar_colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, wear_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
ax.axhline(y=0.70, color='orange', linestyle='--', label='경고(0.70)')
ax.axhline(y=0.85, color='red', linestyle='--', label='교체(0.85)')
ax.set_title('현재 공구 마모 비교', fontweight='bold')
ax.set_ylabel('공구 마모 지수')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
patches = [mpatches.Patch(color='red', label='위험'), mpatches.Patch(color='orange', label='주의'), mpatches.Patch(color='green', label='정상')]
ax.legend(handles=patches, fontsize=9)

# 3. 불량률 추이
ax = axes[1, 0]
for machine, color in colors.items():
    mdf = recent_30[recent_30.machine_id == machine]
    daily = mdf.groupby(mdf.timestamp.dt.date)['quality_pass'].apply(lambda x: (1-x.mean())*100)
    ax.plot(range(len(daily)), daily.values, color=color, label=machine, linewidth=1.5, alpha=0.8)
ax.axhline(y=8, color='red', linestyle='--', alpha=0.7, label='현재 평균 8%')
ax.axhline(y=3, color='green', linestyle='--', alpha=0.7, label='목표 3%')
ax.set_title('일별 불량률 추이 (최근 30일)', fontweight='bold')
ax.set_xlabel('경과 일수')
ax.set_ylabel('불량률 (%)')
ax.legend(fontsize=9)

# 4. RUL 잔여 시간 시각화
ax = axes[1, 1]
rul_hours = [get_cnc_rul(m)['remaining_hours'] for m in machines]
urgencies = [get_cnc_rul(m)['urgency'] for m in machines]
rul_colors = ['red' if u in ['CRITICAL','HIGH'] else ('orange' if u == 'MEDIUM' else 'green') for u in urgencies]
bars = ax.barh(machines, rul_hours, color=rul_colors, edgecolor='black', alpha=0.85)
for bar, val, urg in zip(bars, rul_hours, urgencies):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}h ({urg})', va='center', fontsize=10)
ax.axvline(x=8, color='red', linestyle='--', alpha=0.7, label='위험: 8시간 이내')
ax.set_title('잔여 공구 수명 (시간)', fontweight='bold')
ax.set_xlabel('잔여 시간 (h)')
ax.legend(fontsize=9)

plt.suptitle('CNC 공장 AI 대시보드 — 3대 설비 종합 현황', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('cnc_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('대시보드 저장: cnc_dashboard.png')

## 섹션 7: 내 공장 적용 가이드 (S5 연결)

이 예시를 나의 현장에 어떻게 적용할 수 있을까요?

In [ ]:
apply_guide = """
╔══════════════════════════════════════════════════════════════╗
║          이 예시를 내 공장에 적용하려면?                      ║
╚══════════════════════════════════════════════════════════════╝

Step 1. 데이터 수집 현황 파악
─────────────────────────────
□ 진동/온도/전류 센서 → Track A 방식 이상탐지 적용 가능
□ 공정 파라미터 (속도/압력/온도 등) → Track B 불량 예측 적용 가능
□ 설비 점검 기록 텍스트 → RAG 유지보수 로그 검색 적용 가능
□ 없다면: 저비용 IoT 센서 도입 검토 (KAMP 지원 사업 활용)

Step 2. KAMP 데이터 활용 방법
───────────────────────────────
  1. https://kamp-ai.kr 접속
  2. 좌측 메뉴 → 데이터셋 → 업종 선택 (금속가공/화학/식품 등)
  3. 유사 설비의 공개 데이터셋 다운로드
  4. 이 노트북의 generate_kamp_cnc_data() 부분을 다음으로 교체:
     df = pd.read_csv('kamp_dataset.csv')
  5. 컬럼명 매핑 (mapping_dict 참고)

Step 3. Tool 함수 커스터마이징
────────────────────────────────
  get_cnc_status()     → get_{설비명}_status() 로 변경
  get_cnc_rul()        → 마모 지표명 변경 (tool_wear → 현장 지표명)
  임계값(THRESHOLDS)  → 현장 정상 범위 데이터로 재계산

Step 4. 에이전트 시나리오 작성
────────────────────────────────
  "내 현장에서 가장 자주 받는 질문 3가지"를 테스트 케이스로:
  예) "오늘 2호기 생산 계획에 문제 없을까요?"
  예) "지난주 불량 급증 원인이 뭔가요?"
  예) "다음 정기 정비 일정 언제로 잡아야 할까요?"

Step 5. 파일럿 설계서 작성 (S5 노트북)
──────────────────────────────────────
  → kamp_cnc_pilot_design_template.ipynb 참조
  → 3개월 로드맵 + ROI 계산 + 리스크 관리 계획 포함

───────────────────────────────────────────
도움이 되는 링크:
  KAMP 포털:    https://kamp-ai.kr
  KAMP 데이터:  https://kamp-ai.kr/bigdata (업종별 데이터셋)
  스마트공장:   https://www.smart-factory.kr
───────────────────────────────────────────
"""
print(apply_guide)

In [ ]:
# 컬럼 매핑 예시 (다른 설비/업종 적용 시)
mapping_example = {
    '이 예시 컬럼': 'CNC 공장 → 내 공장 매핑 예시',
    'tool_wear_index': '공구마모 → 금형마모도 / 브레이드마모율 / 베어링마모지수',
    'vibration_x_um': '진동X → 진동센서측정값 / 가속도값_X',
    'cutting_temp_c': '절삭온도 → 가공온도 / 금형온도 / 압출온도',
    'quality_pass': '품질합격 → 제품합격여부 / 검사결과 / OK_NG',
    'spindle_rpm': '스핀들속도 → 금형회전수 / 압출속도 / 컨베이어속도',
    'maintenance_log': '정비로그 → 설비점검일지 / 수리이력 / 작업일보',
}

print('=== 다른 업종 적용 시 컬럼 매핑 예시 ===')
for k, v in mapping_example.items():
    if k != '이 예시 컬럼':
        print(f'  {k:25s} → {v}')

## 다음 단계

| 단계 | 노트북 | 내용 |
|------|--------|------|
| **S5** | `kamp_cnc_pilot_design_template.ipynb` | 파일럿 설계서 작성 |
| **실습** | 내 공장 데이터 + 이 코드 | 현장 적용 |

### 실제 LLM 연동 방법 (실제 에이전트 구현 시)

```python
# OpenAI / Claude API 연동 예시
# X1-S4 기본 노트북 참조: notebooks/01_react_agent_basic.ipynb

from anthropic import Anthropic
client = Anthropic()

tools = [
    {"name": "get_cnc_status",      "description": "...", "input_schema": {...}},
    {"name": "get_cnc_rul",          "description": "...", "input_schema": {...}},
    {"name": "get_quality_prediction","description": "...", "input_schema": {...}},
    {"name": "get_cnc_statistics",   "description": "...", "input_schema": {...}},
    {"name": "get_optimal_parameters","description": "...", "input_schema": {...}},
]
# → X1-S4 기본 노트북 패턴을 이 Tool 함수들에 적용
```

> **KAMP 포털**: https://kamp-ai.kr — 업종별 AI 제조 데이터셋 무료 제공